### 构造互相关运算（卷积）

In [1]:
import torch
from torch import nn

# K: kernel
def corr2d(x ,k) :
    h,w = k.shape
    y = torch.zeros((x.shape[0] - h + 1),(x.shape[1] - w + 1))
    for i in range(y.shape[0]):
        for j in range(y.shape[1] ):
            y[i][j] = (x[i : i+h,j:j+w] * k).sum()
    return y

x = torch.tensor([[0,1,2],[3,4,5],[6,7,8]])
k = torch.tensor([[0,1],[2,3]])

corr2d(x,k)

tensor([[19., 25.],
        [37., 43.]])

### 实现二维卷积层

In [ ]:
class conv2D(nn.Module) :
    def __init__(self, kernelSize):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernelSize))
        self.bias = nn.Parameter(torch.zeros(1))
        
    def forward(self,x):
        return corr2d(x,self.weight) + self.bias


# 简单的边缘检测
x = torch.ones((6,8))
x[:,2:6] = 0
print("x : ",x)

k = torch.tensor([[1,-1]])
k
print("k : ",k)

y = corr2d(x,k)
y
print("y : ",y)

x :  tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])
k :  tensor([[ 1, -1]])
y :  tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])


### 学习 由 `x` 生成 `y` 的卷积核

In [9]:
conv2d = nn.Conv2d(1,1,kernel_size=(1,2),bias=False)

x = x.reshape(1,1,6,8)
y = y.reshape(1,1,6,7)


for i in range(10) :
    y_hat = conv2d(x)
    l = (y_hat - y) ** 2 #lose
    conv2d.zero_grad()
    l.sum().backward()
    conv2d.weight.data[:] -= 3e-2 * conv2d.weight.grad
    if (i + 1) % 2 == 0 :
        print(f'batch {i + 1}, loos : {l.sum() : .3f}')
        

print(conv2d.weight.data.reshape((1,2)))

batch 2, loos :  16.561
batch 4, loos :  5.372
batch 6, loos :  1.964
batch 8, loos :  0.765
batch 10, loos :  0.306
tensor([[ 1.0442, -0.9307]])


### 1*1卷积层

In [2]:
import torch
import torch.nn as nn

# 创建一个输入张量：批量大小=2, 输入通道数=256, 高=14, 宽=14
# 这是一个在中间层很常见的特征图尺寸
input_tensor = torch.randn(2, 256, 14, 14)  # (batch, C_in, H, W)
print("输入张量形状:", input_tensor.shape)

# 定义1x1卷积层：关键就是 kernel_size=1
# 目标：将通道数从256降到64（降维）
conv_1x1 = nn.Conv2d(in_channels=256, out_channels=64, kernel_size=1)

# 执行1x1卷积
output_tensor = conv_1x1(input_tensor)
print("1x1卷积输出形状:", output_tensor.shape)


# 取出第一个样本，位置(i=7, j=7)的所有256个通道值
single_pixel_all_channels = input_tensor[0, :, 7, 7]  # 形状: (256,)
print("单个像素的256个通道值形状:", single_pixel_all_channels.shape)

# 取出第一个1x1卷积核的权重（对应第一个输出通道）
first_1x1_kernel = conv_1x1.weight[0, :, 0, 0]  # 形状: (256,)
print("第一个1x1卷积核权重形状:", first_1x1_kernel.shape)

# 这个输出像素值就是加权求和的结果（加上偏置）
output_value = torch.dot(single_pixel_all_channels, first_1x1_kernel) + conv_1x1.bias[0]
print("手动计算的结果:", output_value.item())

# 验证与直接卷积的结果是否一致
direct_output_value = output_tensor[0, 0, 7, 7]
print("直接卷积的结果:", direct_output_value.item())
print("两者是否接近:", torch.allclose(output_value, direct_output_value))

输入张量形状: torch.Size([2, 256, 14, 14])
1x1卷积输出形状: torch.Size([2, 64, 14, 14])
单个像素的256个通道值形状: torch.Size([256])
第一个1x1卷积核权重形状: torch.Size([256])
手动计算的结果: 0.41581451892852783
直接卷积的结果: 0.41581448912620544
两者是否接近: True
